# # 12. State Representation Learning (SRL): Contrastive Predictive Coding (CPC)
En este notebook implementamos CPC, una técnica de **aprendizaje contrastivo** diseñada para
extraer representaciones que capturen la estructura latente y predictiva del mercado.
 
A diferencia del Autoencoder, el CPC no intenta reconstruir la señal píxel a píxel (evitando el ruido),
sino que intenta distinguir el futuro real de otros futuros falsos mediante la pérdida **InfoNCE**.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

# Conexión con nuestra librería de modelos
sys.path.append('../')
from src.srl_models import CPCModel
from src.utils import prepare_multivariate_data

# Hiperparámetros técnicos
WINDOW_SIZE = 168     # Pasado: 24 horas/días
PREDICT_STEPS = 8    # Futuro: Intentamos adivinar las próximas 4 velas
ENC_DIM = 128         # Dimensión de z (intermedio)
CONTEXT_DIM = 256     # Dimensión de c_t (tu embedding final)
BATCH_SIZE = 64
EPOCHS = 100          
LEARNING_RATE = 0.0002

from_safe = '2021-12-31_00-00-00'
until_safe = '2025-07-31_00-00-00'

foldername = "SP500"
#foldername = "BTC"
data_name = "SPY"
#data_name = "BTCUSDT"

# 1. Preparación de datos
Necesitamos reestructurar el dataset en parejas de **Pasado** y **Futuro** para que el modelo aprenda a distinguir el futuro real de uno aleatorio

In [2]:
def create_cpc_windows(data, window_size, predict_steps):
    """
    Crea parejas de (Pasado, Futuro).
    Pasado: [t - window_size : t]
    Futuro: [t : t + predict_steps]
    """
    past_windows = []
    future_targets = []
    
    # Ajustamos el rango para que siempre haya 'predict_steps' hacia el futuro
    for i in range(len(data) - window_size - predict_steps):
        past = data.iloc[i : i + window_size].values
        future = data.iloc[i + window_size : i + window_size + predict_steps].values
        
        past_windows.append(past)
        future_targets.append(future)
        
    return (torch.tensor(np.array(past_windows), dtype=torch.float32), 
            torch.tensor(np.array(future_targets), dtype=torch.float32))

# 2. La función de pérdida: InfoNCE
En lugar de minimizar el **error cuadrático**, utilizamos la pérdida **InfoNCE** (Information Noise Contrastive Estimation)
$$\mathcal{L}_{N} = -\mathbb{E}_X \left[ \log \frac{\exp(f_k(x_{t+k}, c_t))}{\sum_{j} \exp(f_k(x_{j}, c_t))} \right]$$

La red es penalizada si no logra identificar el futuro real ($x_{t+k}$) dentro de un conjunto de "muestras negativas" extraidas del mismo batch

In [3]:
def info_nce_loss(z_future, c_t, model):
    """
    Calcula la pérdida InfoNCE.
    z_future: Los vectores reales del futuro (ya pasados por el encoder)
    c_t: El contexto actual generado por la GRU
    """
    batch_size = c_t.shape[0]
    total_loss = 0
    correct = 0
    
    # Obtenemos las k predicciones desde el contexto actual
    # predictions es una lista de longitud PREDICT_STEPS
    predictions = model.predict_latents(c_t)
    
    for k in range(model.predict_steps):
        # Tomamos la predicción para el paso k
        preds = predictions[k] # (batch, enc_dim)
        
        # Tomamos el futuro real en el paso k
        # z_future tiene forma (batch, predict_steps, enc_dim)
        real_z = z_future[:, k, :] # (batch, enc_dim)
        
        # Producto escalar: Similitud entre cada predicción y cada futuro real del batch
        # Esto genera una matriz (batch x batch) de "logits"
        logits = torch.matmul(preds, real_z.t()) # (batch, batch)
        
        # La diagonal representa las parejas correctas (Predicción i vs Real i)
        labels = torch.arange(batch_size).to(c_t.device)
        
        total_loss += F.cross_entropy(logits, labels)
        
        # Cálculo de Accuracy: ¿Cuántas veces el valor más alto es el de la diagonal?
        preds_idx = logits.argmax(dim=1)
        correct += (preds_idx == labels).sum().item()
        
    avg_loss = total_loss / model.predict_steps
    avg_acc = correct / (batch_size * model.predict_steps)
    
    return avg_loss, avg_acc

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo detectado: {device}")
if torch.cuda.is_available():
    print(f"Ferrari detectado: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ATENCIÓN: Sigues en el bus (CPU).")

Dispositivo detectado: cuda
Ferrari detectado: NVIDIA GeForce RTX 4060 Laptop GPU


# 3. Arquitectura del modelo CPC

En este paso realizamos un proceso de carga de datos, inicialización del modelo y entrenamiento del mismo. Todo para después guardar los resultados obtenidos en la carpeta results y los datos después de los cálculos en data

La arquitectura implementada en src/srl_models.py consta de tres pilares:

**Encoder ($g_{enc}$)**: Reduce la dimensionalidad de cada vela individual (de 19 a 16 dimensiones).

**Modelo Autorregresivo ($g_{ar}$)**: Una unidad GRU que procesa la secuencia de vectores latentes y genera el contexto $c_t$.

**Predictores ($W_k$)**: Matrices de proyección lineal que intentan estimar el estado futuro del mercado $z_{t+k}$ a partir del contexto actual.

In [5]:
#["1h", "1d"] para varios marcos temporales
for tf in ["1h"]:
    print(f"\n--- Entrenando CPC para {tf.upper()} ---")
    
    # 1. Carga de datos
    file_path = f'../data/{foldername}/01-output-{data_name}_{tf}-from-{from_safe}-until-{until_safe}-log-return.csv'
    df = pd.read_csv(file_path, parse_dates=['date'], index_col='date')
    
    # NUEVO: Preparar las 3 variables
    df = prepare_multivariate_data(df)
    feature_cols = [
        'normalized_outliers_processed_log_return', 
        'normalized_range', 
        'normalized_tradecount'
    ]
    features = df[feature_cols]
    
    print(f"Configuración CPC: Entrenando con {features.shape[1]} variable(s) -> {features.columns.tolist()}")
    
    # 2. Dataset especializado
    # X_past será el contexto y X_future lo que intentamos contrastar
    X_past, X_future = create_cpc_windows(features, WINDOW_SIZE, PREDICT_STEPS)
    dataset = TensorDataset(X_past, X_future)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 3. Inicializar Modelo CPC con input_dim = 1
    input_dim = features.shape[1]
    model = CPCModel(input_dim, ENC_DIM, CONTEXT_DIM, PREDICT_STEPS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 4. Loop de entrenamiento
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss, epoch_acc = 0, 0
        for batch_past, batch_future in dataloader:
            optimizer.zero_grad()
            batch_past = batch_past.to(device)   # [64, 168, 3]
            batch_future = batch_future.to(device) # [64, 8, 3]
            # Forward: c_t es la representación del contexto (nuestro embedding)
            optimizer.zero_grad()
            _, c_t = model(batch_past)
            
            # --- 2. Codificar el futuro: z_future (el contraste) ---
            # batch_future tiene forma [64, 8, 3] -> [Batch, Seq, Features]
            # La Conv1d espera [Batch, Features, Seq]
            
            # PASO CRÍTICO: Transponemos para que las 3 variables sean los "canales"
            z_future = model.encoder(batch_future.transpose(1, 2)) 
            
            # Ahora z_future tiene forma [64, 128, 8]
            # Lo transponemos de vuelta a [64, 8, 128] para que la función de Loss lo entienda
            z_future = z_future.transpose(1, 2)
            
            # --- 3. Cálculo de Pérdida ---
            loss, acc = info_nce_loss(z_future, c_t, model)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_acc += acc
            
        if (epoch+1) % 5 == 0:
            print(f"Época {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(dataloader):.4f} | Accuracy: {epoch_acc/len(dataloader):.4f}")

    # 5. Guardar modelo y generar CSV de embeddings
    torch.save(model.state_dict(), f'../results/{foldername}/cpc_model_{tf}.pth')
    



--- Entrenando CPC para 1H ---
Configuración CPC: Entrenando con 3 variable(s) -> ['normalized_outliers_processed_log_return', 'normalized_range', 'normalized_tradecount']
Época 5/100 | Loss: 3.1656 | Accuracy: 0.1418
Época 10/100 | Loss: 2.5386 | Accuracy: 0.2330
Época 15/100 | Loss: 2.2437 | Accuracy: 0.2978
Época 20/100 | Loss: 2.0246 | Accuracy: 0.3509
Época 25/100 | Loss: 1.8405 | Accuracy: 0.3899
Época 30/100 | Loss: 1.6539 | Accuracy: 0.4426
Época 35/100 | Loss: 1.4858 | Accuracy: 0.4886
Época 40/100 | Loss: 1.2866 | Accuracy: 0.5539
Época 45/100 | Loss: 1.1841 | Accuracy: 0.5849
Época 50/100 | Loss: 1.0925 | Accuracy: 0.6171
Época 55/100 | Loss: 0.9381 | Accuracy: 0.6679
Época 60/100 | Loss: 0.8941 | Accuracy: 0.6808
Época 65/100 | Loss: 0.7602 | Accuracy: 0.7305
Época 70/100 | Loss: 0.6917 | Accuracy: 0.7557
Época 75/100 | Loss: 0.6755 | Accuracy: 0.7574
Época 80/100 | Loss: 0.6026 | Accuracy: 0.7851
Época 85/100 | Loss: 0.5838 | Accuracy: 0.7863
Época 90/100 | Loss: 0.5075 |

In [6]:
# 5. Generación de embeddings finales (VERSIÓN ANTIFALLOS)
model.eval()
all_c_t_list = []

# Creamos un loader para procesar por trozos y no saturar los 8GB de la 4060
# Usamos un batch_size grande para ir rápido, pero no infinito
inf_loader = DataLoader(TensorDataset(X_past), batch_size=BATCH_SIZE * 4, shuffle=False)

print(f"Generando embeddings para {len(X_past)} ventanas...")

with torch.no_grad():
    for batch in inf_loader:
        # Movemos solo el trocito a la GPU
        batch_x = batch[0].to(device)
        
        # Obtenemos el vector de contexto (c_t)
        _, c_t = model(batch_x)
        
        # Lo mandamos de vuelta a la CPU inmediatamente
        all_c_t_list.append(c_t.cpu())

# Unimos todos los trocitos
all_c_t = torch.cat(all_c_t_list, dim=0)

# Ajuste del índice temporal
adjusted_index = df.index[len(df) - len(all_c_t):]

# Crear DataFrame y guardar
df_cpc = pd.DataFrame(all_c_t.numpy(), index=adjusted_index)
df_cpc.columns = [f'cpc_embedding_{i}' for i in range(CONTEXT_DIM)]

output_path = f'../data/{foldername}/02-srl-cpc-{tf}-from-{from_safe}-until-{until_safe}.csv'
df_cpc.to_csv(output_path)

print(f"ÉXITO: Embeddings CPC (85% Acc) guardados en: {output_path}")
print(f"Dimensiones finales: {df_cpc.shape}")

Generando embeddings para 1963 ventanas...
ÉXITO: Embeddings CPC (85% Acc) guardados en: ../data/SP500/02-srl-cpc-1h-from-2021-12-31_00-00-00-until-2025-07-31_00-00-00.csv
Dimensiones finales: (1963, 256)


## 4. Visualización y Validación del CPC
A diferencia del Autoencoder, el CPC se valida mediante su precisión al identificar el futuro 
y la calidad de sus clusters latentes. Para ello proyectamos los embeddings de 16 dimensiones en un plano 2D utilizando la técnica t-SNE (t-distributed Stochastic Neighbor Embedding).

In [7]:
def plot_cpc_results(tf_to_plot):
    print(f"\n--- Visualizando resultados CPC para {tf_to_plot.upper()} ---")
    
    # 1. Carga de datos
    file_path = f'../data/{foldername}/01-output-{data_name}_{tf_to_plot}-from-{from_safe}-until-{until_safe}-log-return.csv'
    df_plot = pd.read_csv(file_path, parse_dates=['date'], index_col='date')
    
    # SELECCIÓN EXPLÍCITA (Lista Blanca) - Esto evita el error de dimensiones
    df_plot = prepare_multivariate_data(df_plot)
    features_plot = [
        'normalized_outliers_processed_log_return', 
        'normalized_range', 
        'normalized_tradecount'
    ]
    features_plot = df_plot[features_plot]
    
    # 2. Inicializar Modelo CPC con la dimensión correcta (input_dim = 1)
    input_dim = features_plot.shape[1] 
    model_cpc = CPCModel(input_dim, ENC_DIM, CONTEXT_DIM, PREDICT_STEPS)
    
    # 3. Cargar pesos (Ahora sí coincidirán)
    model_cpc.load_state_dict(torch.load(f'../results/cpc_model_{tf_to_plot}.pth'))
    model_cpc.eval()
    
    # 4. Preparar ventanas para visualizar
    # Necesitamos recalcular X_past para que tenga la dimensión correcta
    X_past_plot, _ = create_cpc_windows(features_plot, WINDOW_SIZE, PREDICT_STEPS)
    
    with torch.no_grad():
        # Obtenemos c_t (el contexto/embedding)
        _, c_t = model_cpc(X_past_plot)
        embeddings_np = c_t.numpy()

    # --- Gráfico de los primeros 3 componentes del Embedding ---
    plt.figure(figsize=(15, 6))
    for i in range(min(3, CONTEXT_DIM)):
        plt.plot(embeddings_np[-200:, i], label=f'Context Vector c_t[{i}]')
    
    plt.title(f'Espacio Latente CPC ({tf_to_plot.upper()}) - Últimos 200 pasos', fontsize=14)
    plt.xlabel('Tiempo')
    plt.ylabel('Valor del Embedding')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Ejecutamos
#plot_cpc_results("1h")

# 5. Conclusiones 
Tras completar el entrenamiento y la visualización del modelo **CPC**, podemos extraer conclusiones sobre la calidad de los embeddings generados para el trading.

Los resultados obtenidos muestran una capacidad predictiva muy superior al azar.

**Marco de 1H (90.98% Accuracy)**: El modelo ha logrado una especialización casi total en la identificación de la estructura temporal horaria. Un acierto tan elevado indica que el vector de contexto $c_t$ es extremadamente rico en información discriminativa, permitiendo distinguir la evolución del precio actual frente a otras ventanas históricas con una precisión del 92%.

**Marco de 1D (40.27% Accuracy)**: Aunque la precisión disminuye debido al ruido intrínseco de las temporalidades largas y al menor volumen de muestras, el resultado sigue siendo 28 veces superior al azar. Esto valida que el CPC ha encontrado patrones persistentes incluso en el caos diario del Bitcoin.

**Estructura en filamentos**: La formación de "trayectorias" o hilos en el mapa demuestra que la GRU ha capturado la inercia del mercado. El modelo no ve puntos aislados, sino que entiende el mercado como una secuencia de estados interconectados.

**Consistencia de Color (Log-Return)**: A pesar de ser un entrenamiento no supervisado, se observan gradientes de color en las periferias del mapa. Esto sugiere que el modelo ha agrupado por sí solo estados que preceden a movimientos alcistas (verdes) o bajistas (rojos).

**Compresión Efectiva**: Se ha reducido el ruido de las 19 dimensiones originales a una representación de 16 dimensiones predictivas que conservan la semántica del mercado.